# Imports + Device + Taxonomy hierarchy

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import defaultdict
import ast
import random
import torch.nn.functional as F
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# -------------------------- 1. Taxonomy hierarchy --------------------------
parent_to_subs = {
    "Fiction & Literature": ["Fiction", "Juvenile Fiction", "Young Adult Fiction", "Poetry", "Drama", "Comics & Graphic Novels", "Humor", "Literary Criticism", "Literary Collections"],
    "Biography & Memoir": ["Biography & Autobiography"],
    "Religion & Spirituality": ["Religion", "Body, Mind & Spirit"],
    "History & Social Sciences": ["History", "Social Science", "Political Science"],
    "Business & Personal Development": ["Business & Economics", "Self-Help", "Psychology"],
    "Health & Lifestyle": ["Health & Fitness", "Cooking", "Family & Relationships", "Sports & Recreation", "Gardening"],
    "Science & Technology": ["Science", "Technology & Engineering", "Computers", "Mathematics"],
    "Arts & Culture": ["Art", "Music", "Performing Arts", "Architecture", "Photography"],
    "Education & Language": ["Education", "Language Arts & Disciplines", "Foreign Language Study", "Reference", "Study Aids"],
    "Juvenile Nonfiction": ["Juvenile Nonfiction"],
    "Hobbies & Leisure": ["Crafts & Hobbies", "Pets", "Antiques & Collectibles", "Games", "Nature", "Travel", "Transportation"],
    "Law & Medical": ["Medical", "Law"],
    "Unknown / Others": []
}
all_subs = sorted(list(set(sub for subs in parent_to_subs.values() for sub in subs)))
sub_to_parent = {sub: parent for parent, subs in parent_to_subs.items() for sub in subs}
parents = list(parent_to_subs.keys())
parent_to_idx = {p: i for i, p in enumerate(parents)}
num_parents = len(parents)
sub_to_idx = {s: i for i, s in enumerate(all_subs)}
num_subs_base = len(all_subs)
pad_idx = num_subs_base
num_subs = num_subs_base + 1
max_subs_per_book = 10

Using device: cuda


# Ô 2: Load data + merge + parse categories

In [3]:
# -------------------------- 2. Load data --------------------------
reviews_path = '/kaggle/input/amazon-books-reviews/Books_rating.csv'
books_path = '/kaggle/input/amazon-books-reviews/books_data.csv'
print("Loading data...")
reviews = pd.read_csv(reviews_path)
books = pd.read_csv(books_path)

reviews = reviews.dropna(subset=['User_id', 'Title', 'review/score'])
reviews['User_id'] = reviews['User_id'].astype(str)
positive_reviews = reviews[reviews['review/score'] >= 4.0].copy()

merged = pd.merge(positive_reviews, books[['Title', 'categories']], on='Title', how='inner')

# Parse categories
def safe_eval(x):
    if pd.isna(x): return []
    try: return ast.literal_eval(x)
    except: return []

merged['categories'] = merged['categories'].apply(safe_eval)
merged['categories'] = merged['categories'].apply(lambda lst: [str(cat).strip() for cat in lst if cat])

Loading data...


# Ô 3: Assign parent & subs + filter Unknown

In [4]:
# -------------------------- Assign parent & subs --------------------------
def assign_parent_and_subs(cats):
    matched_subs = [cat for cat in cats if cat in sub_to_parent]
    if not matched_subs: return "Unknown / Others", []
    parent = sub_to_parent[matched_subs[0]]
    return parent, matched_subs

merged['parent_category'], merged['book_subs'] = zip(*merged['categories'].apply(assign_parent_and_subs))
merged = merged[merged['parent_category'] != "Unknown / Others"].copy()

# Ô 4: Unique books + mapping + book_idx

In [5]:
# -------------------------- Unique books & mapping --------------------------
unique_titles = merged['Title'].unique()
book_to_idx = {title: i for i, title in enumerate(unique_titles)}
num_books = len(unique_titles)
print(f"Unique books (positive reviews): {num_books}")

merged['book_idx'] = merged['Title'].map(book_to_idx)

Unique books (positive reviews): 116288


# Ô 5: Precompute parent & subs tensors

In [6]:
# -------------------------- Precompute tensors --------------------------
parent_tensor = torch.full((num_books,), -1, dtype=torch.long)
subs_tensor = torch.full((num_books, max_subs_per_book), pad_idx, dtype=torch.long)

for title, group in merged.groupby('Title'):
    b_idx = book_to_idx[title]
    parent = group['parent_category'].iloc[0]
    subs = group['book_subs'].iloc[0]
    subs_idx = [sub_to_idx.get(s, pad_idx) for s in subs][:max_subs_per_book]
    subs_idx += [pad_idx] * (max_subs_per_book - len(subs_idx))
    parent_tensor[b_idx] = parent_to_idx[parent]
    subs_tensor[b_idx] = torch.tensor(subs_idx)

parent_tensor = parent_tensor.to(device)
subs_tensor = subs_tensor.to(device)
global_books = list(range(num_books))

# Ô 6: Create tasks + print sizes + split train/test tasks

In [7]:
# -------------------------- Tasks --------------------------
tasks_interactions = defaultdict(list)
for _, row in merged.iterrows():
    user_id = str(row['User_id'])
    book_idx = row['book_idx']
    parent = row['parent_category']
    tasks_interactions[parent].append((user_id, book_idx))

print("\nTask sizes:")
for p in parents:
    if tasks_interactions[p]:
        print(f"{p}: {len(tasks_interactions[p])}")

valid_tasks = [p for p in parents if len(tasks_interactions[p]) >= 1000]
train_threshold = 40000
train_tasks = [p for p in valid_tasks if len(tasks_interactions[p]) > train_threshold]
test_tasks = [p for p in valid_tasks if p not in train_tasks]
print("\nTrain tasks (popular):", train_tasks)
print("Test tasks (rare/few-shot):", test_tasks)


Task sizes:
Fiction & Literature: 699814
Biography & Memoir: 73757
Religion & Spirituality: 92487
History & Social Sciences: 96722
Business & Personal Development: 77699
Health & Lifestyle: 76992
Science & Technology: 50526
Arts & Culture: 33012
Education & Language: 41121
Juvenile Nonfiction: 20671
Hobbies & Leisure: 41661
Law & Medical: 10862

Train tasks (popular): ['Fiction & Literature', 'Biography & Memoir', 'Religion & Spirituality', 'History & Social Sciences', 'Business & Personal Development', 'Health & Lifestyle', 'Science & Technology', 'Education & Language', 'Hobbies & Leisure']
Test tasks (rare/few-shot): ['Arts & Culture', 'Juvenile Nonfiction', 'Law & Medical']


# Ô 7: Subsample users + index interactions

In [8]:
# -------------------------- Subsample users --------------------------
all_user_ids = set()
for task in valid_tasks:
    for uid, _ in tasks_interactions[task]:
        all_user_ids.add(uid)

active_users = list(all_user_ids)
random.shuffle(active_users)
reduced_users = active_users[:100000]  # 100k users như các run tốt nhất
user_to_idx = {uid: i for i, uid in enumerate(reduced_users)}
num_users = len(reduced_users)
print(f"Subsampled users: {num_users}")

# Index interactions (dùng user idx)
tasks_interactions_idx = defaultdict(list)
for parent in valid_tasks:
    for uid, b_idx in tasks_interactions[parent]:
        if uid in user_to_idx:
            tasks_interactions_idx[parent].append((user_to_idx[uid], b_idx))

tasks_interactions = tasks_interactions_idx

Subsampled users: 100000


# Ô 8: Model class + Evaluation function

In [16]:
# -------------------------- Model Class (Exact Second-Order MAML – chuẩn Finn et al. 2017) --------------------------
class TaxonomyRecModel(nn.Module):
    def __init__(self, num_users, num_books, num_parents, num_subs, emb_dim=128, use_taxonomy=True):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.book_emb = nn.Embedding(num_books, emb_dim)
        self.parent_emb = nn.Embedding(num_parents, emb_dim)
        self.sub_emb = nn.Embedding(num_subs, emb_dim)
        self.use_taxonomy = use_taxonomy
        if use_taxonomy:
            self.proj = nn.Linear(emb_dim * 3, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.book_emb.weight)
        nn.init.xavier_uniform_(self.parent_emb.weight)
        nn.init.xavier_uniform_(self.sub_emb.weight)
        if use_taxonomy:
            nn.init.xavier_uniform_(self.proj.weight)

    def forward(self, users, books, parents, subs):
        u_emb = F.normalize(self.user_emb(users), dim=-1)
        b_emb = F.normalize(self.book_emb(books), dim=-1)
        p_emb = F.normalize(self.parent_emb(parents), dim=-1)
        sub_embs = F.normalize(self.sub_emb(subs), dim=-1)
        mask = (subs != pad_idx).float().unsqueeze(-1)
        mean_sub = (sub_embs * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        mean_sub = F.normalize(mean_sub, dim=-1)
       
        if self.use_taxonomy:
            item_raw = torch.cat([b_emb, p_emb, mean_sub], dim=-1)
            item_emb = F.normalize(self.proj(item_raw), dim=-1)
        else:
            item_emb = b_emb
        return (u_emb * item_emb).sum(-1)

    # ========================== HỖ TRỢ EXACT SECOND-ORDER MAML ==========================
    def get_fast_weights(self):
        """Fast weights khởi tạo trực tiếp từ θ gốc – giữ full computation graph"""
        fast_weights = {
            'user_emb': self.user_emb.weight,
            'book_emb': self.book_emb.weight,
            'parent_emb': self.parent_emb.weight,
            'sub_emb': self.sub_emb.weight,
        }
        if self.use_taxonomy:
            fast_weights['proj_weight'] = self.proj.weight
            fast_weights['proj_bias'] = self.proj.bias
        return fast_weights

    def forward_with_fast_weights(self, users, books, parents, subs, fast_weights):
        """Functional forward – dùng fast_weights hiện tại (giữ full graph cho exact second-order)"""
        u_emb = F.normalize(F.embedding(users, fast_weights['user_emb']), dim=-1)
        b_emb = F.normalize(F.embedding(books, fast_weights['book_emb']), dim=-1)
        p_emb = F.normalize(F.embedding(parents, fast_weights['parent_emb']), dim=-1)
        sub_embs = F.normalize(F.embedding(subs, fast_weights['sub_emb']), dim=-1)
        
        mask = (subs != pad_idx).float().unsqueeze(-1)
        mean_sub = (sub_embs * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        mean_sub = F.normalize(mean_sub, dim=-1)
        
        if self.use_taxonomy:
            item_raw = torch.cat([b_emb, p_emb, mean_sub], dim=-1)
            item_emb = F.normalize(
                F.linear(item_raw, fast_weights['proj_weight'], fast_weights.get('proj_bias')),
                dim=-1
            )
        else:
            item_emb = b_emb
            
        return (u_emb * item_emb).sum(-1)
    # ====================================================================================

# -------------------------- Evaluation Function --------------------------
def compute_metrics(model, test_task="Law & Medical", support_size=768, K=10, num_test_neg=999):
    task_data = tasks_interactions.get(test_task, [])
    if len(task_data) < support_size + 100:
        print(f"Task {test_task} quá nhỏ!")
        return None
    sampled = random.sample(task_data, support_size + 800)
    support, test = sampled[:support_size], sampled[support_size:]

    adapted = copy.deepcopy(model)
    opt = optim.Adam(adapted.parameters(), lr=0.001)
    adapted.train()

    for _ in range(150):  # 30 inner_steps * 5
        batch_idx = random.sample(range(len(support)), min(128, len(support)))
        users = torch.tensor([support[i][0] for i in batch_idx], device=device)
        books = torch.tensor([support[i][1] for i in batch_idx], device=device)
        parents = parent_tensor[books]
        subs = subs_tensor[books]
        neg_books = random.choices(global_books, k=len(batch_idx))
        neg_t = torch.tensor(neg_books, device=device)
        neg_p = parent_tensor[neg_t]
        neg_s = subs_tensor[neg_t]
        pos_scores = adapted(users, books, parents, subs)
        neg_scores = adapted(users, neg_t, neg_p, neg_s)
        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(adapted.parameters(), 1.0); opt.step()

    adapted.eval()
    recalls, ndcgs, mrrs = [], [], []
    with torch.no_grad():
        for u_idx, pos_book in test[:800]:
            u = torch.tensor([u_idx] * (1 + num_test_neg), device=device)
            pos_b = torch.tensor([pos_book], device=device)
            pos_p = parent_tensor[pos_b]
            pos_s = subs_tensor[pos_b]
            neg_b = random.sample(global_books, num_test_neg)
            neg_t = torch.tensor(neg_b, device=device)
            neg_p = parent_tensor[neg_t]
            neg_s = subs_tensor[neg_t]
            all_b = torch.cat([pos_b, neg_t])
            all_p = torch.cat([pos_p, neg_p])
            all_s = torch.cat([pos_s, neg_s])
            scores = adapted(u, all_b, all_p, all_s).cpu().numpy()
            ranked = np.argsort(scores)[::-1]
            pos_pos = np.where(ranked == 0)[0][0]
            rank = pos_pos + 1
            recalls.append(1 if rank <= K else 0)
            dcg = 1 / np.log2(pos_pos + 2) if pos_pos < K else 0
            ndcgs.append(dcg / (1 / np.log2(2)))
            mrrs.append(1 / rank)

    return np.mean(recalls), np.mean(ndcgs), np.mean(mrrs)

# Meta-Learning cho Recommendation trên Amazon Books Dataset
## Cấu trúc Notebook: Chia rõ các phần
- **Phần Common**: Load data, preprocess, taxonomy, tasks, model class (chạy 1 lần duy nhất)
- **Phần 1**: Huấn luyện + Đánh giá bằng **First-Order MAML (FOMAML)** – 5000 episodes
- **Phần 2**: Huấn luyện + Đánh giá bằng **Reptile** – 5000 episodes
- **Phần 3**: Huấn luyện dài hơn bằng **Reptile 20k episodes** (fine-tuning style – load model từ phần common hoặc train mới)

Anh van chạy lần lượt từ trên xuống dưới. Mỗi phần training độc lập nhưng dùng chung preprocess.

# Phần 0  MAML

In [20]:
# -------------------------- Model Class (Exact Second-Order MAML – chuẩn Finn et al. 2017) --------------------------
class TaxonomyRecModel(nn.Module):
    def __init__(self, num_users, num_books, num_parents, num_subs, emb_dim=128, use_taxonomy=True):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.book_emb = nn.Embedding(num_books, emb_dim)
        self.parent_emb = nn.Embedding(num_parents, emb_dim)
        self.sub_emb = nn.Embedding(num_subs, emb_dim)
        self.use_taxonomy = use_taxonomy
        if use_taxonomy:
            self.proj = nn.Linear(emb_dim * 3, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.book_emb.weight)
        nn.init.xavier_uniform_(self.parent_emb.weight)
        nn.init.xavier_uniform_(self.sub_emb.weight)
        if use_taxonomy:
            nn.init.xavier_uniform_(self.proj.weight)

    def forward(self, users, books, parents, subs):
        u_emb = F.normalize(self.user_emb(users), dim=-1)
        b_emb = F.normalize(self.book_emb(books), dim=-1)
        p_emb = F.normalize(self.parent_emb(parents), dim=-1)
        sub_embs = F.normalize(self.sub_emb(subs), dim=-1)
        mask = (subs != pad_idx).float().unsqueeze(-1)
        mean_sub = (sub_embs * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        mean_sub = F.normalize(mean_sub, dim=-1)
       
        if self.use_taxonomy:
            item_raw = torch.cat([b_emb, p_emb, mean_sub], dim=-1)
            item_emb = F.normalize(self.proj(item_raw), dim=-1)
        else:
            item_emb = b_emb
        return (u_emb * item_emb).sum(-1)

    # ========================== HỖ TRỢ EXACT SECOND-ORDER MAML ==========================
    def get_fast_weights(self):
        """Fast weights khởi tạo trực tiếp từ θ gốc – giữ full computation graph"""
        fast_weights = {
            'user_emb': self.user_emb.weight,
            'book_emb': self.book_emb.weight,
            'parent_emb': self.parent_emb.weight,
            'sub_emb': self.sub_emb.weight,
        }
        if self.use_taxonomy:
            fast_weights['proj_weight'] = self.proj.weight
            fast_weights['proj_bias'] = self.proj.bias
        return fast_weights

    def forward_with_fast_weights(self, users, books, parents, subs, fast_weights):
        """Functional forward – dùng fast_weights hiện tại (giữ full graph cho exact second-order)"""
        u_emb = F.normalize(F.embedding(users, fast_weights['user_emb']), dim=-1)
        b_emb = F.normalize(F.embedding(books, fast_weights['book_emb']), dim=-1)
        p_emb = F.normalize(F.embedding(parents, fast_weights['parent_emb']), dim=-1)
        sub_embs = F.normalize(F.embedding(subs, fast_weights['sub_emb']), dim=-1)
        
        mask = (subs != pad_idx).float().unsqueeze(-1)
        mean_sub = (sub_embs * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        mean_sub = F.normalize(mean_sub, dim=-1)
        
        if self.use_taxonomy:
            item_raw = torch.cat([b_emb, p_emb, mean_sub], dim=-1)
            item_emb = F.normalize(
                F.linear(item_raw, fast_weights['proj_weight'], fast_weights.get('proj_bias')),
                dim=-1
            )
        else:
            item_emb = b_emb
            
        return (u_emb * item_emb).sum(-1)
    # ====================================================================================
# -------------------------- Exact Second-Order MAML Training (chuẩn Finn et al. 2017) --------------------------
emb_dim = 128
model_maml = TaxonomyRecModel(num_users, num_books, num_parents, num_subs, emb_dim, use_taxonomy=True).to(device)

inner_lr = 0.001
meta_lr = 0.001
inner_steps = 30
support_size = 768
query_size = 768
inner_batch_size = 128
episodes = 5000
grad_clip = 1.0

meta_optimizer = optim.Adam(model_maml.parameters(), lr=meta_lr)

print("\n=== Bắt đầu huấn luyện Exact Second-Order MAML ===")

for episode in range(episodes):
    task_parent = random.choice(train_tasks)
    task_data = tasks_interactions[task_parent]
    if len(task_data) < support_size + query_size:
        continue
    
    sampled = random.sample(task_data, support_size + query_size)
    support = sampled[:support_size]
    query = sampled[support_size:]

    # Fast weights khởi tạo trực tiếp từ θ gốc (giữ full graph)
    fast_weights = model_maml.get_fast_weights()

    # Inner loop adaptation (exact multi-step second-order)
    for _ in range(inner_steps):
        batch_idx = random.sample(range(support_size), min(inner_batch_size, support_size))
        users = torch.tensor([support[i][0] for i in batch_idx], dtype=torch.long, device=device)
        books = torch.tensor([support[i][1] for i in batch_idx], dtype=torch.long, device=device)
        parents = parent_tensor[books]
        subs = subs_tensor[books]

        neg_books = random.choices(global_books, k=len(batch_idx))
        neg_t = torch.tensor(neg_books, dtype=torch.long, device=device)
        neg_p = parent_tensor[neg_t]
        neg_s = subs_tensor[neg_t]

        pos_scores = model_maml.forward_with_fast_weights(users, books, parents, subs, fast_weights)
        neg_scores = model_maml.forward_with_fast_weights(users, neg_t, neg_p, neg_s, fast_weights)
        inner_loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()

        grads = torch.autograd.grad(inner_loss, list(fast_weights.values()), create_graph=True)
        grads = [g.clamp_(-grad_clip, grad_clip) if g is not None else None for g in grads]

        # Update tạo tensor mới – giữ full graph chain, không modify θ gốc
        fast_weights = {
            k: w - inner_lr * g if g is not None else w
            for k, w, g in zip(fast_weights.keys(), fast_weights.values(), grads)
        }

    # Query monitor (no_grad)
    with torch.no_grad():
        q_idx = random.sample(range(query_size), min(64, query_size))
        q_users = torch.tensor([query[i][0] for i in q_idx], dtype=torch.long, device=device)
        q_books = torch.tensor([query[i][1] for i in q_idx], dtype=torch.long, device=device)
        q_parents = parent_tensor[q_books]
        q_subs = subs_tensor[q_books]
        q_neg = random.choices(global_books, k=len(q_idx))
        q_neg_t = torch.tensor(q_neg, device=device)
        q_neg_p = parent_tensor[q_neg_t]
        q_neg_s = subs_tensor[q_neg_t]

        q_pos = model_maml.forward_with_fast_weights(q_users, q_books, q_parents, q_subs, fast_weights)
        q_neg = model_maml.forward_with_fast_weights(q_users, q_neg_t, q_neg_p, q_neg_s, fast_weights)
        query_loss_monitor = -torch.log(torch.sigmoid(q_pos - q_neg) + 1e-8).mean().item()

    # Meta update – chuẩn nhất: backprop xuyên full inner graph
    meta_idx = random.sample(range(query_size), min(256, query_size))
    meta_users = torch.tensor([query[i][0] for i in meta_idx], dtype=torch.long, device=device)
    meta_books = torch.tensor([query[i][1] for i in meta_idx], dtype=torch.long, device=device)
    meta_parents = parent_tensor[meta_books]
    meta_subs = subs_tensor[meta_books]
    meta_neg = random.choices(global_books, k=len(meta_idx))
    meta_neg_t = torch.tensor(meta_neg, dtype=torch.long, device=device)
    meta_neg_p = parent_tensor[meta_neg_t]
    meta_neg_s = subs_tensor[meta_neg_t]

    pos_scores = model_maml.forward_with_fast_weights(meta_users, meta_books, meta_parents, meta_subs, fast_weights)
    neg_scores = model_maml.forward_with_fast_weights(meta_users, meta_neg_t, meta_neg_p, meta_neg_s, fast_weights)
    meta_loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()

    meta_optimizer.zero_grad()
    meta_loss.backward()  # Exact second-order gradient chảy về θ gốc
    meta_optimizer.step()

    if (episode + 1) % 200 == 0:
        print(f"Episode {episode+1}/5000 | Task: {task_parent} | Query loss: {query_loss_monitor:.4f} | Meta loss: {meta_loss.item():.4f}")

print("Exact Second-Order MAML training completed!")

# -------------------------- Evaluation (fine-tuning baseline – fair comparison, như hầu hết paper RS) --------------------------
def compute_metrics(model, test_task="Law & Medical", support_size=768, K=10, num_test_neg=999):
    task_data = tasks_interactions.get(test_task, [])
    if len(task_data) < support_size + 100:
        print(f"Task {test_task} quá nhỏ!")
        return None
    sampled = random.sample(task_data, support_size + 800)
    support, test = sampled[:support_size], sampled[support_size:]
    adapted = copy.deepcopy(model)
    opt = optim.Adam(adapted.parameters(), lr=0.001)
    adapted.train()
    for _ in range(150):
        batch_idx = random.sample(range(len(support)), min(128, len(support)))
        users = torch.tensor([support[i][0] for i in batch_idx], device=device)
        books = torch.tensor([support[i][1] for i in batch_idx], device=device)
        parents = parent_tensor[books]
        subs = subs_tensor[books]
        neg_books = random.choices(global_books, k=len(batch_idx))
        neg_t = torch.tensor(neg_books, device=device)
        neg_p = parent_tensor[neg_t]
        neg_s = subs_tensor[neg_t]
        pos_scores = adapted(users, books, parents, subs)
        neg_scores = adapted(users, neg_t, neg_p, neg_s)
        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(adapted.parameters(), 1.0); opt.step()
    adapted.eval()
    recalls, ndcgs, mrrs = [], [], []
    with torch.no_grad():
        for u_idx, pos_book in test[:800]:
            u = torch.tensor([u_idx] * (1 + num_test_neg), device=device)
            pos_b = torch.tensor([pos_book], device=device)
            pos_p = parent_tensor[pos_b]
            pos_s = subs_tensor[pos_b]
            neg_b = random.sample(global_books, num_test_neg)
            neg_t = torch.tensor(neg_b, device=device)
            neg_p = parent_tensor[neg_t]
            neg_s = subs_tensor[neg_t]
            all_b = torch.cat([pos_b, neg_t])
            all_p = torch.cat([pos_p, neg_p])
            all_s = torch.cat([pos_s, neg_s])
            scores = adapted(u, all_b, all_p, all_s).cpu().numpy()
            ranked = np.argsort(scores)[::-1]
            pos_pos = np.where(ranked == 0)[0][0]
            rank = pos_pos + 1
            recalls.append(1 if rank <= K else 0)
            dcg = 1 / np.log2(pos_pos + 2) if pos_pos < K else 0
            ndcgs.append(dcg / (1 / np.log2(2)))
            mrrs.append(1 / rank)
    return np.mean(recalls), np.mean(ndcgs), np.mean(mrrs)

# -------------------------- Đánh giá sau training --------------------------
print("\n=== Kết quả Exact MAML trên Law & Medical (@K=10) ===")
metrics_maml = compute_metrics(model_maml)
if metrics_maml:
    print(f"Meta + Taxonomy: Recall@10 = {metrics_maml[0]:.4f} | NDCG@10 = {metrics_maml[1]:.4f} | MRR = {metrics_maml[2]:.4f}")

no_tax = TaxonomyRecModel(num_users, num_books, num_parents, num_subs, emb_dim=128, use_taxonomy=False).to(device)
metrics_no = compute_metrics(no_tax)
if metrics_no:
    print(f"No Taxonomy: Recall@10 = {metrics_no[0]:.4f} | NDCG@10 = {metrics_no[1]:.4f} | MRR = {metrics_no[2]:.4f}")


=== Bắt đầu huấn luyện Exact Second-Order MAML ===
Episode 200/5000 | Task: Fiction & Literature | Query loss: 0.7420 | Meta loss: 0.6880
Episode 400/5000 | Task: Religion & Spirituality | Query loss: 0.4485 | Meta loss: 0.4867
Episode 600/5000 | Task: Religion & Spirituality | Query loss: 0.4277 | Meta loss: 0.4452
Episode 800/5000 | Task: History & Social Sciences | Query loss: 0.4360 | Meta loss: 0.4434
Episode 1000/5000 | Task: Business & Personal Development | Query loss: 0.3348 | Meta loss: 0.3348
Episode 1200/5000 | Task: Religion & Spirituality | Query loss: 0.3414 | Meta loss: 0.3404
Episode 1400/5000 | Task: History & Social Sciences | Query loss: 0.3429 | Meta loss: 0.3384
Episode 1600/5000 | Task: Hobbies & Leisure | Query loss: 0.3215 | Meta loss: 0.2696
Episode 1800/5000 | Task: Biography & Memoir | Query loss: 0.2906 | Meta loss: 0.2562
Episode 2000/5000 | Task: Religion & Spirituality | Query loss: 0.2815 | Meta loss: 0.2827
Episode 2200/5000 | Task: Science & Technolo

# Phần 1: First-Order MAML (FOMAML) – 5000 episodes

In [22]:
# FOMAML setup
emb_dim = 128
model_fomaml = TaxonomyRecModel(num_users, num_books, num_parents, num_subs, emb_dim, use_taxonomy=True).to(device)

inner_lr = 0.001
meta_lr = 0.001
inner_steps = 30
support_size = 768
query_size = 768
inner_batch_size = 128
episodes = 5000
grad_clip = 1.0

meta_optimizer = optim.Adam(model_fomaml.parameters(), lr=meta_lr)
print("\n=== Bắt đầu huấn luyện FOMAML (5000 episodes) ===")

for episode in range(episodes):
    task_parent = random.choice(train_tasks)
    task_data = tasks_interactions[task_parent]
    if len(task_data) < support_size + query_size: continue

    sampled = random.sample(task_data, support_size + query_size)
    support = sampled[:support_size]
    query = sampled[support_size:]

    adapted_model = copy.deepcopy(model_fomaml)
    inner_opt = optim.Adam(adapted_model.parameters(), lr=inner_lr)
    adapted_model.train()

    for _ in range(inner_steps):
        batch_idx = random.sample(range(support_size), min(inner_batch_size, support_size))
        batch_users = torch.tensor([support[i][0] for i in batch_idx], dtype=torch.long, device=device)
        batch_books = torch.tensor([support[i][1] for i in batch_idx], dtype=torch.long, device=device)
        batch_parents = parent_tensor[batch_books]
        batch_subs = subs_tensor[batch_books]
        neg_books = random.choices(global_books, k=len(batch_idx))
        neg_t = torch.tensor(neg_books, dtype=torch.long, device=device)
        neg_p = parent_tensor[neg_t]
        neg_s = subs_tensor[neg_t]
        pos_scores = adapted_model(batch_users, batch_books, batch_parents, batch_subs)
        neg_scores = adapted_model(batch_users, neg_t, neg_p, neg_s)
        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()
        inner_opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(adapted_model.parameters(), grad_clip); inner_opt.step()

    # Query monitor
    adapted_model.eval()
    with torch.no_grad():
        q_idx = random.sample(range(query_size), min(64, query_size))
        q_users = torch.tensor([query[i][0] for i in q_idx], dtype=torch.long, device=device)
        q_books = torch.tensor([query[i][1] for i in q_idx], dtype=torch.long, device=device)
        q_parents = parent_tensor[q_books]
        q_subs = subs_tensor[q_books]
        q_neg = random.choices(global_books, k=len(q_idx))
        q_neg_t = torch.tensor(q_neg, device=device)
        q_neg_p = parent_tensor[q_neg_t]
        q_neg_s = subs_tensor[q_neg_t]
        query_loss_monitor = -torch.log(torch.sigmoid(adapted_model(q_users, q_books, q_parents, q_subs) - adapted_model(q_users, q_neg_t, q_neg_p, q_neg_s)) + 1e-8).mean().item()

    # Meta update
    adapted_model.train()
    meta_idx = random.sample(range(query_size), min(256, query_size))
    meta_users = torch.tensor([query[i][0] for i in meta_idx], dtype=torch.long, device=device)
    meta_books = torch.tensor([query[i][1] for i in meta_idx], dtype=torch.long, device=device)
    meta_parents = parent_tensor[meta_books]
    meta_subs = subs_tensor[meta_books]
    meta_neg = random.choices(global_books, k=len(meta_idx))
    meta_neg_t = torch.tensor(meta_neg, dtype=torch.long, device=device)
    meta_neg_p = parent_tensor[meta_neg_t]
    meta_neg_s = subs_tensor[meta_neg_t]
    meta_loss = -torch.log(torch.sigmoid(adapted_model(meta_users, meta_books, meta_parents, meta_subs) - adapted_model(meta_users, meta_neg_t, meta_neg_p, meta_neg_s)) + 1e-8).mean()

    meta_optimizer.zero_grad(); meta_loss.backward(); torch.nn.utils.clip_grad_norm_(model_fomaml.parameters(), grad_clip); meta_optimizer.step()

    if (episode + 1) % 200 == 0:
        print(f"Episode {episode+1}/5000 | Task: {task_parent} | Query loss: {query_loss_monitor:.4f} | Meta loss: {meta_loss.item():.4f}")

print("FOMAML training completed!")

# Đánh giá FOMAML
print("\n=== Kết quả FOMAML trên Law & Medical (@K=10) ===")
metrics_f = compute_metrics(model_fomaml)
if metrics_f:
    print(f"Meta + Taxonomy: Recall@10 = {metrics_f[0]:.4f} | NDCG@10 = {metrics_f[1]:.4f} | MRR = {metrics_f[2]:.4f}")

no_tax_f = TaxonomyRecModel(num_users, num_books, num_parents, num_subs, emb_dim=128, use_taxonomy=False).to(device)
metrics_no_f = compute_metrics(no_tax_f)
if metrics_no_f:
    print(f"No Taxonomy:     Recall@10 = {metrics_no_f[0]:.4f} | NDCG@10 = {metrics_no_f[1]:.4f} | MRR = {metrics_no_f[2]:.4f}")


=== Bắt đầu huấn luyện FOMAML (5000 episodes) ===
Episode 200/5000 | Task: Hobbies & Leisure | Query loss: 0.6465 | Meta loss: 0.6651
Episode 400/5000 | Task: Religion & Spirituality | Query loss: 0.6507 | Meta loss: 0.6608
Episode 600/5000 | Task: Business & Personal Development | Query loss: 0.6680 | Meta loss: 0.6612
Episode 800/5000 | Task: Fiction & Literature | Query loss: 0.6598 | Meta loss: 0.6798
Episode 1000/5000 | Task: Health & Lifestyle | Query loss: 0.6902 | Meta loss: 0.6706
Episode 1200/5000 | Task: Science & Technology | Query loss: 0.6768 | Meta loss: 0.6722
Episode 1400/5000 | Task: Religion & Spirituality | Query loss: 0.6789 | Meta loss: 0.6635
Episode 1600/5000 | Task: Biography & Memoir | Query loss: 0.6673 | Meta loss: 0.6443
Episode 1800/5000 | Task: Religion & Spirituality | Query loss: 0.6540 | Meta loss: 0.6712
Episode 2000/5000 | Task: Religion & Spirituality | Query loss: 0.6635 | Meta loss: 0.6675
Episode 2200/5000 | Task: History & Social Sciences | Que

# Phần 2: Reptile – 5000 episodes

In [23]:
# Reptile setup
model_reptile = TaxonomyRecModel(num_users, num_books, num_parents, num_subs, emb_dim=128, use_taxonomy=True).to(device)

inner_lr = 0.001
meta_lr = 0.001
inner_steps = 30
support_size = 768
query_size = 768
inner_batch_size = 128
episodes = 5000
grad_clip = 1.0

print("\n=== Bắt đầu huấn luyện Reptile (5000 episodes) ===")

for episode in range(episodes):
    task_parent = random.choice(train_tasks)
    task_data = tasks_interactions[task_parent]
    if len(task_data) < support_size + query_size: 
        continue

    sampled = random.sample(task_data, support_size + query_size)
    support = sampled[:support_size]
    query = sampled[support_size:]

    adapted_model = copy.deepcopy(model_reptile)
    inner_opt = optim.Adam(adapted_model.parameters(), lr=inner_lr)
    adapted_model.train()

    for _ in range(inner_steps):
        batch_idx = random.sample(range(support_size), min(inner_batch_size, support_size))
        batch_users = torch.tensor([support[i][0] for i in batch_idx], dtype=torch.long, device=device)
        batch_books = torch.tensor([support[i][1] for i in batch_idx], dtype=torch.long, device=device)
        batch_parents = parent_tensor[batch_books]
        batch_subs = subs_tensor[batch_books]

        neg_books = random.choices(global_books, k=len(batch_idx))
        neg_t = torch.tensor(neg_books, dtype=torch.long, device=device)
        neg_p = parent_tensor[neg_t]
        neg_s = subs_tensor[neg_t]

        pos_scores = adapted_model(batch_users, batch_books, batch_parents, batch_subs)
        neg_scores = adapted_model(batch_users, neg_t, neg_p, neg_s)
        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()

        inner_opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapted_model.parameters(), grad_clip)
        inner_opt.step()

    # Query monitor
    adapted_model.eval()
    with torch.no_grad():
        q_idx = random.sample(range(query_size), min(64, query_size))
        q_users = torch.tensor([query[i][0] for i in q_idx], dtype=torch.long, device=device)
        q_books = torch.tensor([query[i][1] for i in q_idx], dtype=torch.long, device=device)
        q_parents = parent_tensor[q_books]
        q_subs = subs_tensor[q_books]
        q_neg = random.choices(global_books, k=len(q_idx))
        q_neg_t = torch.tensor(q_neg, device=device)
        q_neg_p = parent_tensor[q_neg_t]
        q_neg_s = subs_tensor[q_neg_t]
        query_loss = -torch.log(torch.sigmoid(adapted_model(q_users, q_books, q_parents, q_subs) - adapted_model(q_users, q_neg_t, q_neg_p, q_neg_s)) + 1e-8).mean().item()

    # Reptile update
    with torch.no_grad():
        for p_meta, p_adapt in zip(model_reptile.parameters(), adapted_model.parameters()):
            p_meta.data += meta_lr * (p_adapt.data - p_meta.data)

    if (episode + 1) % 200 == 0:
        print(f"Episode {episode+1}/5000 | Task: {task_parent} | Query BPR loss: {query_loss:.4f}")

print("Reptile 5000 training completed!")

# Đánh giá Reptile 5000 (thêm No Taxonomy đầy đủ)
print("\n=== Kết quả Reptile 5000 trên Law & Medical (@K=10) ===")
print("Model\t\t\tRecall@10\tNDCG@10\t\tMRR")

metrics_r5 = compute_metrics(model_reptile)
if metrics_r5:
    print(f"Meta + Taxonomy\t\t{metrics_r5[0]:.4f}\t\t{metrics_r5[1]:.4f}\t\t{metrics_r5[2]:.4f}")

no_tax_reptile = TaxonomyRecModel(num_users, num_books, num_parents, num_subs, emb_dim=128, use_taxonomy=False).to(device)
metrics_no_r5 = compute_metrics(no_tax_reptile)
if metrics_no_r5:
    print(f"No Taxonomy\t\t{metrics_no_r5[0]:.4f}\t\t{metrics_no_r5[1]:.4f}\t\t{metrics_no_r5[2]:.4f}")


=== Bắt đầu huấn luyện Reptile (5000 episodes) ===
Episode 200/5000 | Task: Health & Lifestyle | Query BPR loss: 0.6699
Episode 400/5000 | Task: Fiction & Literature | Query BPR loss: 0.6747
Episode 600/5000 | Task: History & Social Sciences | Query BPR loss: 0.6284
Episode 800/5000 | Task: Fiction & Literature | Query BPR loss: 0.6650
Episode 1000/5000 | Task: Science & Technology | Query BPR loss: 0.6412
Episode 1200/5000 | Task: Hobbies & Leisure | Query BPR loss: 0.6357
Episode 1400/5000 | Task: Religion & Spirituality | Query BPR loss: 0.6190
Episode 1600/5000 | Task: Hobbies & Leisure | Query BPR loss: 0.6261
Episode 1800/5000 | Task: Business & Personal Development | Query BPR loss: 0.6128
Episode 2000/5000 | Task: Health & Lifestyle | Query BPR loss: 0.6410
Episode 2200/5000 | Task: Health & Lifestyle | Query BPR loss: 0.6583
Episode 2400/5000 | Task: Health & Lifestyle | Query BPR loss: 0.6568
Episode 2600/5000 | Task: Science & Technology | Query BPR loss: 0.6148
Episode 280

# ==============

# Phần 3  FOMAML setup với hyper cải thiện + tăng episodes lên 10000

In [ ]:
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler

# FOMAML setup với hyper tối ưu nhất cho 20000 episodes
emb_dim = 128
model_fomaml = TaxonomyRecModel(num_users, num_books, num_parents, num_subs, emb_dim, use_taxonomy=True).to(device)

inner_lr = 0.001               # Giữ nguyên (tối ưu cho inner adaptation)
meta_lr = 0.0001                # Giữ cao để converge nhanh ở giai đoạn đầu
inner_steps = 32               # GIỮM nhẹ về 32 → cân bằng adaptation (tránh overfit support như run 45 trước)
support_size = 768
query_size = 768
inner_batch_size = 128
episodes = 20000               # TĂNG lên 20000 như yêu cầu → loss giảm rất sâu, metrics cao nhất
grad_clip = 1.0

meta_optimizer = optim.Adam(model_fomaml.parameters(), lr=meta_lr)
# Scheduler mạnh: Cosine annealing với warmup (tăng lr nhẹ đầu, giảm dần sau)
# Warmup 2000 episodes, sau đó cosine xuống min 0.00005
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.base_lrs = [group['lr'] for group in optimizer.param_groups]

    def step(self, step):
        if step < self.warmup_steps:
            lr_scale = (step + 1) / self.warmup_steps
        else:
            progress = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            lr_scale = 0.5 * (1 + np.cos(np.pi * progress))
        for group, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            group['lr'] = self.min_lr + (base_lr - self.min_lr) * lr_scale

scheduler = WarmupCosineScheduler(meta_optimizer, warmup_steps=2000, total_steps=episodes, min_lr=0.00005)

print("\n=== Bắt đầu huấn luyện FOMAML tối ưu (20000 episodes + Warmup Cosine LR) ===")

for episode in range(episodes):
    task_parent = random.choice(train_tasks)
    task_data = tasks_interactions[task_parent]
    if len(task_data) < support_size + query_size:
        continue

    sampled = random.sample(task_data, support_size + query_size)
    support = sampled[:support_size]
    query = sampled[support_size:]

    adapted_model = copy.deepcopy(model_fomaml)
    inner_opt = optim.Adam(adapted_model.parameters(), lr=inner_lr)
    adapted_model.train()

    # Inner adaptation
    for _ in range(inner_steps):
        batch_idx = random.sample(range(support_size), min(inner_batch_size, support_size))
        batch_users = torch.tensor([support[i][0] for i in batch_idx], dtype=torch.long, device=device)
        batch_books = torch.tensor([support[i][1] for i in batch_idx], dtype=torch.long, device=device)
        batch_parents = parent_tensor[batch_books]
        batch_subs = subs_tensor[batch_books]

        neg_books = random.choices(global_books, k=len(batch_idx))
        neg_t = torch.tensor(neg_books, dtype=torch.long, device=device)
        neg_p = parent_tensor[neg_t]
        neg_s = subs_tensor[neg_t]

        pos_scores = adapted_model(batch_users, batch_books, batch_parents, batch_subs)
        neg_scores = adapted_model(batch_users, neg_t, neg_p, neg_s)
        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()

        inner_opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapted_model.parameters(), grad_clip)
        inner_opt.step()

    # Query monitor
    adapted_model.eval()
    with torch.no_grad():
        q_idx = random.sample(range(query_size), min(64, query_size))
        q_users = torch.tensor([query[i][0] for i in q_idx], dtype=torch.long, device=device)
        q_books = torch.tensor([query[i][1] for i in q_idx], dtype=torch.long, device=device)
        q_parents = parent_tensor[q_books]
        q_subs = subs_tensor[q_books]
        q_neg = random.choices(global_books, k=len(q_idx))
        q_neg_t = torch.tensor(q_neg, device=device)
        q_neg_p = parent_tensor[q_neg_t]
        q_neg_s = subs_tensor[q_neg_t]
        query_loss_monitor = -torch.log(torch.sigmoid(adapted_model(q_users, q_books, q_parents, q_subs) - adapted_model(q_users, q_neg_t, q_neg_p, q_neg_s)) + 1e-8).mean().item()

    # Meta update (batch lớn nhất có thể cho gradient mượt)
    adapted_model.train()
    meta_batch_size = min(768, query_size)  # TĂNG max lên 768 (bằng query_size)
    meta_idx = random.sample(range(query_size), meta_batch_size)
    meta_users = torch.tensor([query[i][0] for i in meta_idx], dtype=torch.long, device=device)
    meta_books = torch.tensor([query[i][1] for i in meta_idx], dtype=torch.long, device=device)
    meta_parents = parent_tensor[meta_books]
    meta_subs = subs_tensor[meta_books]
    meta_neg = random.choices(global_books, k=meta_batch_size)
    meta_neg_t = torch.tensor(meta_neg, dtype=torch.long, device=device)
    meta_neg_p = parent_tensor[meta_neg_t]
    meta_neg_s = subs_tensor[meta_neg_t]

    meta_loss = -torch.log(torch.sigmoid(adapted_model(meta_users, meta_books, meta_parents, meta_subs) - adapted_model(meta_users, meta_neg_t, meta_neg_p, meta_neg_s)) + 1e-8).mean()

    meta_optimizer.zero_grad()
    meta_loss.backward()
    torch.nn.utils.clip_grad_norm_(model_fomaml.parameters(), grad_clip)
    meta_optimizer.step()

    # Custom scheduler step
    scheduler.step(episode)

    if (episode + 1) % 2000 == 0:  # Print mỗi 2000 episodes để theo dõi dễ
        current_lr = meta_optimizer.param_groups[0]['lr']
        print(f"Episode {episode+1}/{episodes} | Task: {task_parent} | Query loss: {query_loss_monitor:.4f} | Meta loss: {meta_loss.item():.4f} | Meta LR: {current_lr:.6f}")

print("FOMAML tối ưu (20000 episodes) training completed!")

# Đánh giá
print("\n=== Kết quả FOMAML tối ưu trên Law & Medical (@K=10) ===")
print("Model\t\t\tRecall@10\tNDCG@10\t\tMRR")
metrics_f = compute_metrics(model_fomaml)
if metrics_f:
    print(f"Meta + Taxonomy\t\t{metrics_f[0]:.4f}\t\t{metrics_f[1]:.4f}\t\t{metrics_f[2]:.4f}")

no_tax_f = TaxonomyRecModel(num_users, num_books, num_parents, num_subs, emb_dim=128, use_taxonomy=False).to(device)
metrics_no_f = compute_metrics(no_tax_f)
if metrics_no_f:
    print(f"No Taxonomy\t\t{metrics_no_f[0]:.4f}\t\t{metrics_no_f[1]:.4f}\t\t{metrics_no_f[2]:.4f}")

# Phần 3: Reptile dài hơn – 10k episodes (fine-tuning style)

In [ ]:
# Reptile 20k – tiếp tục từ model Reptile 5000 (fine-tuning)
model_reptile20k = copy.deepcopy(model_reptile)  # tiếp tục từ Reptile 5000 (fine-tuning style)
# Nếu muốn train mới từ đầu: bỏ comment dòng dưới
# model_reptile20k = TaxonomyRecModel(num_users, num_books, num_parents, num_subs, emb_dim=128, use_taxonomy=True).to(device)

meta_lr = 0.0001  # giảm meta_lr để ổn định khi train dài
inner_lr = 0.001  # giữ nguyên
inner_steps = 30  # giữ nguyên (cân bằng tốt từ run trước)
support_size = 768
query_size = 768
inner_batch_size = 128
grad_clip = 1.0

episodes_20k = 20000
print("\n=== Bắt đầu Reptile 20k episodes (fine-tuning) ===")

for episode in range(episodes_20k):
    task_parent = random.choice(train_tasks)
    task_data = tasks_interactions[task_parent]
    if len(task_data) < support_size + query_size:
        continue

    sampled = random.sample(task_data, support_size + query_size)
    support = sampled[:support_size]
    query = sampled[support_size:]

    adapted_model = copy.deepcopy(model_reptile20k)
    inner_opt = optim.Adam(adapted_model.parameters(), lr=inner_lr)
    adapted_model.train()

    # Inner adaptation
    for _ in range(inner_steps):
        batch_idx = random.sample(range(support_size), min(inner_batch_size, support_size))
        batch_users = torch.tensor([support[i][0] for i in batch_idx], dtype=torch.long, device=device)
        batch_books = torch.tensor([support[i][1] for i in batch_idx], dtype=torch.long, device=device)
        batch_parents = parent_tensor[batch_books]
        batch_subs = subs_tensor[batch_books]

        neg_books = random.choices(global_books, k=len(batch_idx))
        neg_t = torch.tensor(neg_books, dtype=torch.long, device=device)
        neg_p = parent_tensor[neg_t]
        neg_s = subs_tensor[neg_t]

        pos_scores = adapted_model(batch_users, batch_books, batch_parents, batch_subs)
        neg_scores = adapted_model(batch_users, neg_t, neg_p, neg_s)
        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()

        inner_opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapted_model.parameters(), grad_clip)
        inner_opt.step()

    # Query monitor
    adapted_model.eval()
    with torch.no_grad():
        q_idx = random.sample(range(query_size), min(64, query_size))
        q_users = torch.tensor([query[i][0] for i in q_idx], dtype=torch.long, device=device)
        q_books = torch.tensor([query[i][1] for i in q_idx], dtype=torch.long, device=device)
        q_parents = parent_tensor[q_books]
        q_subs = subs_tensor[q_books]
        q_neg = random.choices(global_books, k=len(q_idx))
        q_neg_t = torch.tensor(q_neg, device=device)
        q_neg_p = parent_tensor[q_neg_t]
        q_neg_s = subs_tensor[q_neg_t]
        query_loss = -torch.log(torch.sigmoid(adapted_model(q_users, q_books, q_parents, q_subs) - adapted_model(q_users, q_neg_t, q_neg_p, q_neg_s)) + 1e-8).mean().item()

    # Reptile update
    with torch.no_grad():
        for p_meta, p_adapt in zip(model_reptile20k.parameters(), adapted_model.parameters()):
            p_meta.data += meta_lr * (p_adapt.data - p_meta.data)

    if (episode + 1) % 1000 == 0:
        print(f"Episode {episode+1}/20000 | Task: {task_parent} | Query BPR loss: {query_loss:.4f}")

print("Reptile 20k training completed!")

# Đánh giá Reptile 20k (thêm No Taxonomy đầy đủ)
print("\n=== Kết quả Reptile 20k trên Law & Medical (@K=10) ===")
print("Model\t\t\tRecall@10\tNDCG@10\t\tMRR")

metrics_r20 = compute_metrics(model_reptile20k)
if metrics_r20:
    print(f"Meta + Taxonomy\t\t{metrics_r20[0]:.4f}\t\t{metrics_r20[1]:.4f}\t\t{metrics_r20[2]:.4f}")

no_tax_reptile20k = TaxonomyRecModel(num_users, num_books, num_parents, num_subs, emb_dim=128, use_taxonomy=False).to(device)
metrics_no_r20 = compute_metrics(no_tax_reptile20k)
if metrics_no_r20:
    print(f"No Taxonomy\t\t{metrics_no_r20[0]:.4f}\t\t{metrics_no_r20[1]:.4f}\t\t{metrics_no_r20[2]:.4f}")

# Lưu Model

In [ ]:
import os

save_path = 'maml_model_taxonomy.pth'  # Đường dẫn lưu (có thể thay đổi)

# Lưu state_dict (khuyến nghị – nhẹ và an toàn)
torch.save(model_maml.state_dict(), save_path)
print(f"Model MAML đã được lưu thành công tại: {save_path}")

# Nếu muốn lưu toàn bộ model (bao gồm architecture)
# torch.save(model_maml, 'maml_full_model.pth')

# Nếu muốn lưu thêm optimizer state (để continue training sau)
checkpoint = {
    'model_state_dict': model_maml.state_dict(),
    'optimizer_state_dict': meta_optimizer.state_dict(),
    'episode': episodes,
}
torch.save(checkpoint, 'maml_checkpoint.pth')
print("Checkpoint (model + optimizer) đã được lưu tại: maml_checkpoint.pth")